# Figure S6 — Integrated annotation of cNMF programs across hiPSC–arterial EC differentiation

Canonical source inspected: the D0–D3 integrated annotation notebooks in `programDE/notebooks`. This port aligns three layers for each day: programDE perturbation effects, consensus cNMF loadings, and motif enrichment. Regulators, marker genes and motifs are unioned within day.


In [ ]:
# Repository-path bootstrap for local support modules.
from pathlib import Path
import sys


def _find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "cc_perturbseq").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing "
        "src/cc_perturbseq. Run this notebook from within the "
        "cc-perturbseq-manuscript-analysis checkout."
    )


REPO_ROOT = _find_repo_root()
for _support_path in (REPO_ROOT, REPO_ROOT / "src"):
    _support_path_text = str(_support_path)
    if _support_path_text not in sys.path:
        sys.path.insert(0, _support_path_text)

print(
    "Repository support paths added:",
    REPO_ROOT,
    "and",
    REPO_ROOT / "src",
)


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from src.cc_perturbseq.figure_notebook import (
    get_path, initialize_figure, load_figure_manifest, panel_directory, write_receipt,
)
from src.cc_perturbseq.program_panels import *

def export_figure(fig, stem, panel):
    out = panel_directory(ctx, panel)
    paths = {}
    for suffix, kwargs in [("pdf", {}), ("svg", {}), ("png", {"dpi": 300})]:
        path = out / f"{stem}.{suffix}"
        fig.savefig(path, **kwargs)
        paths[suffix] = str(path)
    return paths

FIGURE = load_figure_manifest("Figure_S06")
ctx = initialize_figure(FIGURE)
ctx["input_audit"]


## Load common inputs


In [ ]:
effect_path = get_path(ctx["paths"], "programde.effect_matrix_all_days")
q_path = get_path(ctx["paths"], "programde.qvalue_matrix_all_days", required=False)
effects = load_program_effects(effect_path, q_path, filename=config_value(ctx["paths"], "programde.effects_filename", required=False))
spectra_path = get_path(ctx["paths"], "ccperturbseq.cnmf_spectra")
spectra = load_spectra(spectra_path)
annotation_path = get_path(ctx["paths"], "ccperturbseq.program_annotations", required=False)
labels, annotated_by_day = load_program_annotations(annotation_path)
long_motif_path = get_path(ctx["paths"], "multiome.enhancer_enrichment_results", required=False)
long_motif_path = long_motif_path if long_motif_path and long_motif_path.exists() else None
motifs = load_motif_enrichment(long_motif_path, fdr_template=config_value(ctx["paths"], "multiome.motif_fdr_template", required=False), score_template=config_value(ctx["paths"], "multiome.motif_score_template", required=False))


## Day-grouped integrated annotation

The default program set is the curated set used in the temporal-map notebooks. Replace it with `annotated_by_day` only after the final program annotation table carries an explicit figure-inclusion field.


In [ ]:
PROGRAMS_BY_DAY = {
    "D0": [7, 22, 29],
    "D1": [28, 13, 30, 48, 33],
    "D2": [15, 14, 17],
    "D3": [2, 3, 8, 11, 24, 27, 34, 35],
}
TARGETS_BY_DAY = {
    day: select_top_targets(effects, day, programs, n_per_program=4)
    for day, programs in PROGRAMS_BY_DAY.items()
}
fig, day_tables = plot_multi_day_annotation(
    effects, spectra, motifs,
    programs_by_day=PROGRAMS_BY_DAY,
    targets_by_day=TARGETS_BY_DAY,
    program_labels=labels,
)
outputs = export_figure(fig, "Figure_S06_integrated_program_annotation", "all")
flat_tables = {f"{day}_{name}": table for day, tables in day_tables.items() for name, table in tables.items()}
source = save_source_tables(flat_tables, panel_directory(ctx, "all"), "Figure_S06")
write_receipt(ctx, "all", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"effects": str(effect_path), "spectra": str(spectra_path), "motifs": str(long_motif_path), "annotations": str(annotation_path)}, outputs={**outputs, **source}, parameters={"programs_by_day": PROGRAMS_BY_DAY, "targets_by_day": TARGETS_BY_DAY})
plt.show()
